# Deriving the Linear Kalman Filter

*Course 2 — Linear Kalman Filter, Part 2. Specializes the [six generic SPI steps](07_Sequential_Probabilistic_Inference_Six_Steps.ipynb) to a **linear** model. Every expectation now becomes a concrete matrix formula you can code directly.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Linear Model and Noise Assumptions

- Assume a discrete-time linear state-space model:

$$
x_k = A_{k-1}x_{k-1} + B_{k-1}u_{k-1} + w_{k-1}, \qquad z_k = C_k x_k + D_k u_k + v_k.
$$

  → State propagates linearly with a known input and additive process noise; the measurement is a linear function of state and input plus sensor noise.

- Noise assumptions: $w_k$, $v_k$ are **zero-mean, white, mutually uncorrelated** Gaussians:

$$
\mathbb{E}[w_n w_k^T] = \Sigma_{\tilde{w}}\delta_{nk}, \qquad \mathbb{E}[v_n v_k^T] = \Sigma_{\tilde{v}}\delta_{nk}, \qquad \mathbb{E}[w_n v_k^T]=0.
$$

  → Each noise is uncorrelated in time and with the other; today's noise tells you nothing about any other instant.

- **→ Intuition:** because everything is linear and all inputs are Gaussian, **all densities stay Gaussian forever** — so the mean+covariance the SPI recursion tracks are the *complete* description, and the six steps are *exact*, not approximate. (The assumptions are rarely perfectly true in practice, but the method works remarkably well anyway.)

### 🧩 Prediction Step 1a — State Prediction

- Compute the predicted state (take the conditional expectation of the linear dynamics):

$$
\hat{x}_k^- = \mathbb{E}[A_{k-1}x_{k-1} + B_{k-1}u_{k-1} + w_{k-1}\mid\mathbb{Z}_{k-1}] = A_{k-1}\hat{x}_{k-1}^{+} + B_{k-1}u_{k-1}.
$$

  → Push the **previous best estimate** through the model and add the known input. The noise term $w_{k-1}$ drops out because it is zero-mean. This is directly implementable in code.

- **→ Intuition:** with no new measurement yet, the best forecast is "run the model forward from where we last were."

---

### 🧩 Prediction Step 1b — Prediction-Error Covariance

- Propagate the uncertainty (the Lyapunov step from [06](06_Stochastic_Processes_and_Propagating_Uncertainty.ipynb)):

$$
\Sigma_{\tilde{x},k}^{-} = A_{k-1}\,\Sigma_{\tilde{x},k-1}^{+}\,A_{k-1}^{T} + \Sigma_{\tilde{w}}.
$$

  → The old uncertainty is reshaped by the dynamics ($A\Sigma A^T$) and then **grown** by fresh process noise ($+\Sigma_{\tilde w}$). Cross terms vanish because $w_{k-1}$ is uncorrelated with the state.

- **→ Intuition:** for a *stable* system $A\Sigma A^T$ is **contractive** (uncertainty about a decaying state shrinks), while $\Sigma_{\tilde{w}}$ always **adds** uncertainty. Prediction alone makes us *less* certain overall — measurements (step 2) are what claw certainty back.

### 🧩 Prediction Step 1c — Output Prediction

- Predict what the sensor should read:

$$
\hat{z}_k = \mathbb{E}[C_k x_k + D_k u_k + v_k \mid \mathbb{Z}_{k-1}] = C_k\hat{x}_k^{-} + D_k u_k.
$$

  → Map the predicted state through the output equation; $v_k$ drops out (zero-mean). $\hat z_k$ is our best guess of the measurement *before* we see it.

- **→ Intuition:** we now have a prediction *and* an expectation for the sensor. The gap between $\hat z_k$ and the real $z_k$ (the innovation) is the fuel for the correction.

### 🧩 Correction Step 2a — The Kalman Gain

- First the **innovation covariance** (uncertainty of $\tilde z_k = z_k - \hat z_k = C_k\tilde x_k^- + v_k$):

$$
\Sigma_{\tilde{z},k} = C_k\,\Sigma_{\tilde{x},k}^{-}\,C_k^{T} + \Sigma_{\tilde{v}}.
$$

  → Uncertainty in the predicted output = state uncertainty seen through $C$ ($C\Sigma^- C^T$) plus raw sensor noise ($\Sigma_{\tilde v}$).

- Then the **state–output cross-covariance** and the **Kalman gain**:

$$
\Sigma_{\tilde{x}\tilde{z},k} = \Sigma_{\tilde{x},k}^{-}\,C_k^{T}, \qquad
L_k = \Sigma_{\tilde{x}\tilde{z},k}\,\Sigma_{\tilde{z},k}^{-1} = \Sigma_{\tilde{x},k}^{-}C_k^{T}\big(C_k\Sigma_{\tilde{x},k}^{-}C_k^{T}+\Sigma_{\tilde{v}}\big)^{-1}.
$$

  → The gain is **(how state uncertainty couples to the output) ÷ (total output uncertainty)**. Big when the state is uncertain and the sensor is clean (trust the measurement); small when the state is already well known or the sensor is noisy (trust the model).

- **→ Intuition:** $L_k$ is the single most important quantity in Kalman filtering — the entire point of tracking covariances is to compute this optimal, time-varying blending factor. It adapts every step to current conditions.

### 🧩 Correction Steps 2b & 2c — Update State and Covariance

- **2b — state update** using the innovation:

$$
\hat{x}_k^{+} = \hat{x}_k^{-} + L_k\,(z_k - \hat{z}_k).
$$

  → Nudge the prediction toward the data by gain × surprise. Zero surprise ⇒ no change; big surprise with high gain ⇒ big correction.

- **2c — covariance update** (three equivalent forms):

$$
\Sigma_{\tilde{x},k}^{+} = \Sigma_{\tilde{x},k}^{-} - L_k\,\Sigma_{\tilde{z},k}\,L_k^{T} = \Sigma_{\tilde{x},k}^{-} - L_k C_k \Sigma_{\tilde{x},k}^{-} = (I - L_k C_k)\,\Sigma_{\tilde{x},k}^{-}.
$$

  → The measurement **shrinks** the covariance (we subtract a positive-semidefinite amount). We are now more certain than before the measurement.

- **→ Intuition:** if a measurement is skipped/rejected, set $L_k = 0$: then $\hat x_k^+ = \hat x_k^-$ and $\Sigma^+ = \Sigma^-$ — the filter simply coasts on the prediction. (Used later for [fault rejection](10_KF_Extensions_FaultDetection_SteadyState_Smoothing.ipynb).)

### 🧩 The Complete Linear Kalman Filter

Run once per sample interval $k$:

**Prediction**
$$
\textbf{1a: } \hat{x}_k^- = A_{k-1}\hat{x}_{k-1}^+ + B_{k-1}u_{k-1}
$$
$$
\textbf{1b: } \Sigma_{\tilde{x},k}^- = A_{k-1}\Sigma_{\tilde{x},k-1}^+ A_{k-1}^T + \Sigma_{\tilde{w}}
$$
$$
\textbf{1c: } \hat{z}_k = C_k\hat{x}_k^- + D_k u_k
$$

**Correction**
$$
\textbf{2a: } L_k = \Sigma_{\tilde{x},k}^- C_k^T\big(C_k\Sigma_{\tilde{x},k}^- C_k^T + \Sigma_{\tilde{v}}\big)^{-1}
$$
$$
\textbf{2b: } \hat{x}_k^+ = \hat{x}_k^- + L_k\big(z_k - \hat{z}_k\big)
$$
$$
\textbf{2c: } \Sigma_{\tilde{x},k}^+ = (I - L_k C_k)\,\Sigma_{\tilde{x},k}^-
$$

Reference Octave loop (matches the Boot-Camp implementation):

```octave
for k = 2:length(t)
  % --- Prediction ---
  xhat   = Ad*xhat + Bd*u(:,k-1);          % 1a
  SigmaX = Ad*SigmaX*Ad' + SigmaW;         % 1b
  zhat   = Cd*xhat + Dd*u(:,k);            % 1c
  % --- Correction ---
  L      = SigmaX*Cd' / (Cd*SigmaX*Cd' + SigmaV);   % 2a
  xhat   = xhat + L*(z(:,k) - zhat);       % 2b
  SigmaX = SigmaX - L*Cd*SigmaX;           % 2c
  % store xhat and 3*sqrt(diag(SigmaX)) for bounds
end
```

- **→ Intuition:** notice the recursion is **linear** and needs only the previous $\hat x^+$ and $\Sigma^+$ — no growing history buffer. That $\mathcal{O}(1)$-memory recursion is what made the Kalman filter revolutionary.

### 🧩 Summary

- For a linear model, the six SPI steps become **explicit matrix equations** — no approximation needed, because linearity keeps every density Gaussian.

- **Prediction** propagates mean ($A\hat x^+ + Bu$) and covariance ($A\Sigma^+A^T + \Sigma_{\tilde w}$) and forecasts the output ($C\hat x^- + Du$).

- **Correction** forms the gain $L_k = \Sigma^-C^T(C\Sigma^-C^T+\Sigma_{\tilde v})^{-1}$, updates the state with the innovation, and shrinks the covariance to $(I-L_kC)\Sigma^-$.

- The filter outputs both an estimate and calibrated **confidence bounds** $\pm 3\sqrt{\operatorname{diag}(\Sigma^+)}$, using only the previous step's results.

- Next we make this robust and realistic: numerical safety, tuning, biases, and correlated noise.

---
*Next: [09 · Making the KF Bulletproof](09_Making_the_KF_Bulletproof.ipynb).*